# Chapter 4: FastAI — Maximum Results, Minimum Code

### Why This Matters

**FastAI** sits on top of PyTorch (much like Keras sits on top of TensorFlow) and is built around one central philosophy: apply well-researched best practices (transfer learning, learning rate scheduling, data augmentation) *automatically* by default, so a small amount of code produces a strong result. It's especially strong for computer vision and works well for tabular data too.

In [ ]:
from fastai.tabular.all import *

cat_names = ["contract", "internet", "tech_support"]
cont_names = ["tenure", "monthly_charges", "tickets", "senior"]

splits = RandomSplitter(valid_pct=0.2)(range_of(df))
to = TabularPandas(df, procs=[Categorify, FillMissing, Normalize],
                    cat_names=cat_names, cont_names=cont_names,
                    y_names="churn", splits=splits)

dls = to.dataloaders(bs=64)
learn = tabular_learner(dls, metrics=accuracy)
learn.fit_one_cycle(5, 1e-2)

Compare this to the manual preprocessing you'd otherwise write by hand (Volume 05's `pd.get_dummies` and `StandardScaler`): `TabularPandas` with `procs=[Categorify, FillMissing, Normalize]` handles categorical encoding, missing-value imputation, and numeric scaling in one declarative line. `fit_one_cycle` uses the **1-cycle learning rate policy** (a research-validated technique that varies the learning rate up and then down over training) as its default training method, rather than a single fixed learning rate — this alone often meaningfully improves both training speed and final accuracy compared to naive defaults, and you get it automatically without needing to understand or implement it yourself.

For image classification specifically, FastAI's signature workflow is **transfer learning** — starting from a model already trained on millions of general images (like `resnet18`) and fine-tuning it on your specific, much smaller dataset:

In [ ]:
from fastai.vision.all import *

path = untar_data(URLs.PETS)                                       # a sample dataset, downloaded automatically
dls = ImageDataLoaders.from_name_func(path, get_image_files(path), get_label_from_filename)
learn = vision_learner(dls, resnet18, metrics=accuracy)
learn.fine_tune(3)                                                    # 3 epochs of fine-tuning on your data

**Transfer learning is arguably the single most practically important deep learning technique for any team without Google/Meta-scale data and compute:** training an image classifier truly from scratch typically needs millions of labeled images; fine-tuning a pretrained model like `resnet18` can produce a strong, business-usable classifier from a few hundred or thousand labeled examples of your specific problem, because the pretrained model has already learned general-purpose visual features (edges, textures, shapes) that transfer to almost any new image task.

### Cheat Sheet — FastAI

| Task | Code |
|---|---|
| Tabular data | `TabularPandas(df, procs=[Categorify, FillMissing, Normalize], ...)` |
| Tabular model | `tabular_learner(dls, metrics=accuracy)` |
| Image transfer learning | `vision_learner(dls, resnet18, metrics=accuracy)` |
| Train (best-practice default) | `learn.fit_one_cycle(n)` or `learn.fine_tune(n)` |

---